# Content

In this notebook, we will train a Character level RNN.

Character level means that instead of generating complete WORDS(or a complete word at a time) like in a word level RNN, our model will learn to generate individual characters(one single character at a time).

This approach have a few benefits over word level RNN, like:

1. Smaller vocabulary size: Instead of our vocabulary size consisting of all unique word of the corpus, it simply consists of unique characters
2. Easier tokenization: The tokenization simply consists of breaking down the text into characters

For our model, we will use the Tiny Shakespeare dataset. It consists of a single text file containing a selection of works by William Shakespeare, specifically his plays.


In [ ]:
from IPython.display import clear_output

In [ ]:
# %pip install numpy matplotlib torch tqdm requests

clear_output()

In [ ]:
import random
import requests

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm

## Loading the data

In [ ]:
# Load the dataset
dataset_url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
shakespeare_text = requests.get(dataset_url).text

In [ ]:
# Create a mapping from character to index and vice versa

unique_chars = sorted(list(set(shakespeare_text)))
vocab_size = len(unique_chars)
char_to_idx = {char: idx for idx, char in enumerate(unique_chars)}
idx_to_char = {idx: char for idx, char in enumerate(unique_chars)}

In [ ]:
# Encode the dataset into integer indices

encoded_sp_text = torch.Tensor([char_to_idx[char] for char in tqdm(shakespeare_text)]).type(torch.long)

In [ ]:
encoded_sp_text.shape

## Defining the dataset class

In [ ]:
class ShakeSpeareCharactersDataset(Dataset):

    def __init__(self, encoded_sp_text, seq_len=100):

        self.encoded_sp_text = encoded_sp_text
        self.seq_len = seq_len

    def __len__(self):
        # We will randomly sample parts of text in this dataset and return them so we don't have a specific length, so we can define it anything.
        # Note that this length will be used as number of samples per epoch. Increasing this will increase the number of samples processed per epoch.
        return 5000

    def __getitem__(self, idx):

        start_idx = random.randint(0, len(self.encoded_sp_text) - self.seq_len)
        end_idx = start_idx + self.seq_len

        # calling torch.Tensor on a value that's already possibly tensor because:
        # 1. If it's not a tensor, it'll be converted to one
        # 2. If it's already a tensor, It's safer to make a copy of it and this will do just that.
        text = torch.Tensor(self.encoded_sp_text[start_idx:end_idx])
        return text


In [ ]:
train_dataset = ShakeSpeareCharactersDataset(encoded_sp_text, seq_len=200)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

## Defining the model

In [ ]:
class CharRNN(nn.Module):

    def __init__(self, vocab_size, hidden_size, num_layers):
        super(CharRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embed = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        x = self.embed(x)
        out, hidden = self.rnn(x, hidden)
        out = out.reshape(out.size(0) * out.size(1), self.hidden_size)
        out = self.fc(out)
        return out, hidden

    def init_hidden(self, batch_size):
        return torch.zeros(self.num_layers, batch_size, self.hidden_size)

## Training the model

In [ ]:
# Hyperparameters
hidden_size = 256
num_layers = 3
num_epochs = 200
learning_rate = 5e-4

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
# Model, Loss, Optimizer
model = CharRNN(vocab_size, hidden_size, num_layers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
train_losses = []

In [ ]:
model.train()
model.to(device)

In [ ]:
# Training loop
for epoch in range(num_epochs):

    for encoded_text_batch in tqdm(train_loader):

        # .shape[0] for batch size. cannot use batch_size variable because actual batch can be smaller(when not enough elements left to make full batch)
        hidden = model.init_hidden(encoded_text_batch.shape[0]).to(device)

        inputs, targets = encoded_text_batch[..., :-1], encoded_text_batch[..., 1:]
        inputs, targets = inputs.to(device), targets.to(device)

        # Forward pass
        outputs, hidden = model(inputs, hidden)
        loss = criterion(outputs, targets.reshape(-1))

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 1 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}')
        train_losses.append(loss.item())

In [ ]:
plt.plot(train_losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.show()

In [ ]:
model.cpu()

## Testing the model

In [ ]:
def generate_text(model, start_str, length):
    model.eval()
    hidden = model.init_hidden(1)
    input = torch.tensor([[char_to_idx[char] for char in start_str]], dtype=torch.long)

    with torch.no_grad():

        generated_str = start_str
        for _ in range(length):
            output, hidden = model(input, hidden)
            char_idx = torch.argmax(output[-1]).item()
            char = idx_to_char[char_idx]
            generated_str += char
            input = torch.tensor([[char_idx]], dtype=torch.long)

    return generated_str

In [ ]:
# Example generation
start_str = "Is th"
generated_text = generate_text(model, start_str, 300)
print(generated_text)

In [ ]:
# Example generation
start_str = "Charm"
generated_text = generate_text(model, start_str, 300)
print(generated_text)